In [5]:
from data_pipeline.fetch_all_team_data import fetch_all_team_data

"""
main jupyter notebook for Ml project]
Ronan kenny 
23404724
"""

'\nmain jupyter notebook for Ml project]\nRonan kenny \n23404724\n'

In [6]:

# use module to load in the total team data and extract match statistics for all teams
all_match_stats = fetch_all_team_data()



Fetching all teams available...
Fetching all team stats for team_id: 34bd3e4e...
Fetching all team stats for team_id: 94c0327a...
Fetching all team stats for team_id: 67d88578...
Fetching all team stats for team_id: 165b36e5...
Fetching all team stats for team_id: bf499d89...
Fetching all team stats for team_id: 246228fc...
Fetching all team stats for team_id: d7f92ae2...
Fetching all team stats for team_id: f13dc547...
Fetching all team stats for team_id: 4409215c...
Fetching all team stats for team_id: d0cffe5e...
Fetching all team stats for team_id: b523256e...
Fetching all team stats for team_id: e9c9c61d...
Fetching all team stats for team_id: 4ec76284...
Fetching all team stats for team_id: 29dd316b...
Fetching all team stats for team_id: ad805c09...
Fetching all team stats for team_id: 94b19a71...
Fetching all team stats for team_id: a659d6d1...
Fetching all team stats for team_id: 7aca7d09...
Fetching all team stats for team_id: 18b82d15...
Fetching all team stats for team_id: 

In [11]:
from data_pipeline.consolidate_data_fields import consolidate_data_fields

# use helper function to consolodate duplicate stats 
# that may be stored in different names but represent the same data
all_match_stats = consolidate_data_fields(all_match_stats)

In [12]:
from data_pipeline.filter_rows_with_required_data import filter_rows_with_required_data

required_columns = [
    "tries",
    "kicks_from_hand",
    #"kick_percent_success",
    #"territory",
    #"possession",
    #"22m_entries",
]

model_candidate_data = filter_rows_with_required_data(all_match_stats, required_columns)



In [13]:

from data_pipeline.create_target_col import create_target_col
filtered_dataset = create_target_col(model_candidate_data)


In [23]:

import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.dummy import DummyClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression


# update string date values to date object for filtering and split
filtered_dataset['game_date'] = pd.to_datetime(filtered_dataset['game_date'], format='%Y%m%d')
filtered_dataset['year'] = filtered_dataset['game_date'].dt.year


# define sci-kit learn pipeline 
# my dataset is missing some values so I will use imputer for these values in the pipeline - prevents leakagfe
dummy_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('model', DummyClassifier(strategy='prior'))
])

logreg_pipeline = Pipeline(steps =[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
    ('model' , LogisticRegression(max_iter=1000))
])

# potential feature cols
extended_feature_columns = [
    "kicks_from_hand",
    "kick_percent_success",
    "territory",
    "possession",
    "22m_entries",
]

#feature_cols = ["kicks_from_hand"] # main training feature

feature_cols = extended_feature_columns

years = sorted(filtered_dataset['year'].unique())
min_train_years = 3
                                      

In [24]:
# walk forward validation split training loop
import numpy as np
from sklearn.metrics import roc_auc_score

# will run loop incrementing the next year to be used for validation / testing 
#mimicks how data is collected irl over time and improved upon 

for i in range(min_train_years, len(years)):
    train_years = years[:i]   
    test_year = years[i]      
    
    #split the dataframe based on the years
    train_df = filtered_dataset[filtered_dataset['year'].isin(train_years)]
    test_df = filtered_dataset[filtered_dataset['year'] == test_year]
    
    # Separate the features from the target answers 
    X_train = train_df[feature_cols]
    y_train = train_df['high_tries_scored']
    X_test = test_df[feature_cols]
    y_test = test_df['high_tries_scored']
    
    print(f"\n FOLD: Train {train_years} -> Test {test_year}")
    print(f"  Train: {len(train_df)} rows | Test: {len(test_df)} rows")

    # find most recent date in the current loop iterations training data
    anchor_date = train_df['game_date'].max()

    # recency based weighting 
    
    # calculate how many days ago each game was played
    days_ago = (anchor_date - train_df['game_date']).dt.days
    
    # create the weights - games from a year ago get half the weight
    weights = np.exp(-(np.log(2) / 365) * days_ago)


    # training 

    # dummy model 
    dummy_pipeline.fit(X_train, y_train, model__sample_weight=weights)
    dummy_preds = dummy_pipeline.predict_proba(X_test)[:, 1]
    dummy_auc = roc_auc_score(y_test, dummy_preds)
    print(f" Dummy ROC-AUC: {dummy_auc:.3f}")

    logreg_pipeline.fit(X_train, y_train, model__sample_weight= weights)
    logreg_preds = logreg_pipeline.predict_proba(X_test)[:,1]
    logreg_auc = roc_auc_score(y_test, logreg_preds)
    
    print(f" LogReg AUC: {logreg_auc:.3f}")


 FOLD: Train [np.int32(2022), np.int32(2023), np.int32(2024)] -> Test 2025
  Train: 4204 rows | Test: 1033 rows
 Dummy ROC-AUC: 0.500
 LogReg AUC: 0.604

 FOLD: Train [np.int32(2022), np.int32(2023), np.int32(2024), np.int32(2025)] -> Test 2026
  Train: 5237 rows | Test: 828 rows
 Dummy ROC-AUC: 0.500
 LogReg AUC: 0.611
